TODO:
   - [x] перевести формирование счетчика навыков в функцию
   - [ ] добавить обьединенную сортировку по заработной плате
   - [ ] на основе вышеприведенных пунктов: добавить просмотр навыков по 0.XX самым высокооплачиваемым вакансиям
   - [x] try - except блок при загрузке данных с сайта
   - [ ] сделать приложением?

In [1]:
import os
import requests
import time
from tqdm.notebook import tqdm
#from datetime import datetime
#from dateutil.parser import parse

import re
from collections import Counter

import polars as pl
#import pandas as pd

In [2]:
%matplotlib inline

In [3]:
BREAK_STOP_LEVEL = 3
DATA_PATH = os.path.join('.', 'data')

In [4]:
addr = 'https://api.hh.ru/vacancies'
head = {'User-Agent': 'NoApp StudyPro/0.9.1'}

In [5]:
vacancies_list = ['Data scientist',
                  'Data science',
                  'Дата Саентист',
                  'Machine learning',
                  'ML',
                  'Ml-engeneer',
                  'CV-engeneer',
                  'NLP',
                 ]

'area': {'id': '1', 'name': 'Москва'...}    
'salary': {'from': 300000, 'to': None, 'currency': 'RUR', 'gross': False}    
'description': {'......'}    
'key_skills': [{'name': 'Мат стат'}, {'name': 'Мат анализ'}, {'name': 'Python'}, {'name': 'Git'}]     
'published_at': YYYY-MM-DDThh:mm:ss±hhmm

In [6]:
class UtilityClass():
    """
    """
    def __init__(self):
        self._clr = lambda x: (re.sub(r'[<>.,_+*?!/()]', ' ', str(x))).strip().lower()


    def _grade(self, inp_vac_name: str) -> str:
        """
        """
        name = self._clr(inp_vac_name)

        if 'стажер' in name or\
           'стажёр' in name or\
           'intern' in name:
            return 'intern'

        if 'junior' in name or\
           'младший' in name:
            return 'junior'

        if 'middle' in name:
            return 'middle'

        if 'старший' in name or\
           'senior' in name:
            return 'senior'

        if 'head' in name or\
           'директор' in name or\
           'руководитель' in name:
            return 'head'

        if 'leader' in name or\
           'лидер' in name:
            return 'team-leader'

        return 'unknown' # middle?


    def _salary(self, inp_salary: dict) -> tuple:
        """
        """
        ret_from = -1
        ret_to = -1
        ret_currency = '-1'
        if 'from' in inp_salary.keys() and\
            not isinstance(inp_salary['from'], type(None)):
            ret_from = inp_salary['from']

        if 'to' in inp_salary.keys() and\
            not isinstance(inp_salary['to'], type(None)):
            ret_to = inp_salary['to']

        if 'currency' in inp_salary.keys() and\
            not isinstance(inp_salary['currency'], type(None)):
            ret_currency = inp_salary['currency']

        return (ret_from, ret_to, ret_currency)



class VacancyClass(UtilityClass):
    """
    """
    def __init__(self):
        super().__init__()
        self.__all_id = set()
        self.__new_id = list()
        self.__area = list()
        self.__date_created = list()
        self.__date_published = list()
        self.__descr = list()
        self.__experience = list()
        self.__role = list()
        self.__salary_from = list()
        self.__salary_to = list()
        self.__salary_cur = list()
        self.__vac_name = list()
        self.__url = list()

        if os.path.exists(os.path.join(DATA_PATH, 'vacancies.csv')):
            tmp_df = pl.read_csv(os.path.join(DATA_PATH, 'vacancies.csv'), columns=['vacancy_id'])
            self.__all_id = set(tmp_df.unique().to_numpy().reshape(-1))
            print('all_id ', len(self.__all_id))


    def add_id(self, inp_id: int) -> None:
        """
        """
        self.__all_id.add(inp_id)


    def check_id(self, inp_id: int) -> bool:
        """
        """
        return inp_id in self.__all_id


    def collect_data(self, inp_vacancy: dict) -> None:
        """
        """
        self.__new_id.append(int(inp_vacancy['id']))
        self.__vac_name.append(inp_vacancy['name'].lower())
        descr = self._clr(inp_vacancy['description'])
        self.__descr.append(descr)
        self.__role.append(inp_vacancy['professional_roles'][0]['name'])
        self.__experience.append(inp_vacancy['experience']['id'])
        self.__date_created.append(inp_vacancy['created_at'])
        self.__date_published.append(inp_vacancy['published_at'])
        self.__url.append(inp_vacancy['alternate_url'])
        self.__area.append(inp_vacancy['area']['name'])

        if isinstance(inp_vacancy['salary'], type(None)):
            self.__salary_from.append(-1)
            self.__salary_to.append(-1)
            self.__salary_cur.append('-1')
        else:
            (s_from, s_to, s_cur) = self._salary(inp_vacancy['salary'])
            self.__salary_from.append(s_from)
            self.__salary_to.append(s_to)
            self.__salary_cur.append(s_cur)


    def reset(self) -> None:
        """
        """
        self.__new_id = list()
        self.__area = list()
        self.__date_created = list()
        self.__date_published = list()
        self.__descr = list()
        self.__experience = list()
        self.__role = list()
        self.__salary_from = list()
        self.__salary_to = list()
        self.__salary_cur = list()
        self.__vac_name = list()
        self.__url = list()


    def savevacancies(self) -> None:
        """
        """
        if len(self.__new_id) <= 0:
            return 

        new_data = pl.DataFrame({
                    'vacancy_id': self.__new_id,
                    'vacancy_name': self.__vac_name,
                    'role': self.__role,
                    'experience': self.__experience,
                    'date_created': self.__date_created,
                    'date_published': self.__date_published,
                    'salary_from': self.__salary_from,
                    'salary_to': self.__salary_to,
                    'salary_currency': self.__salary_cur,
                    'url': self.__url,
                    'area': self.__area,
                    'description': self.__descr,
                         })
        new_data = new_data.with_columns(
            pl.col('vacancy_name').map_elements(self._grade, return_dtype=pl.String).\
                alias('grade'),
            #pl.col('vacancy_id').cast(pl.Int64)),
        )
        #change column order for better view
        new_data = new_data.select(['vacancy_id', 'vacancy_name', 'role', 'grade', 'experience',
                                    'date_created', 'date_published', 
                                    'salary_from', 'salary_to', 'salary_currency',
                                    'url', 'area', 'description'
                                   ])

        if os.path.exists(os.path.join(DATA_PATH, 'vacancies.csv')):
            data = pl.read_csv(os.path.join(DATA_PATH, 'vacancies.csv'))
            data = pl.concat([data, new_data])
        else:
            data = new_data


        data.write_csv(os.path.join(DATA_PATH, 'vacancies.csv'))



class VacancySkillsClass(UtilityClass):
    """
    """
    def __init__(self):
        super().__init__()
        self.__id = list()
        self.__name = list()
        self.__key_skills = list()
        self.__date_created = list()
        self.__date_published = list()


    def collect_skills(self, inp_vacancy: dict) -> None:
        """
        """
        if len(inp_vacancy['key_skills']) == 0:
            return

        for skill in inp_vacancy['key_skills']:
            self.__key_skills.append(skill['name'].lower())
            self.__id.append(int(inp_vacancy['id']))
            self.__name.append(inp_vacancy['name'].lower())
            self.__date_created.append(inp_vacancy['created_at'])
            self.__date_published.append(inp_vacancy['published_at'])


    def reset(self) -> None:
        """
        """
        self.__id = list()
        self.__name = list()
        self.__key_skills = list()
        self.__date_created = list()
        self.__date_published = list()


    def saveskills(self) -> None:
        """
        """
        if len(self.__id) <= 0:
            return

        new_data = pl.DataFrame({
                    'vacancy_id': self.__id,
                    'vacancy_name': self.__name,
                    'key_skills': self.__key_skills,
                    'date_created': self.__date_created,
                    'date_published': self.__date_published,
                         })
        new_data = new_data.with_columns(
            pl.col('vacancy_name').map_elements(self._grade, return_dtype=pl.String).\
                alias('grade'),
            #pl.col('vacancy_id').cast(pl.Int64)),
            )
        #change column order for better view
        new_data = new_data.select(['vacancy_id', 'vacancy_name', 'grade',
                                   'key_skills', 'date_created', 'date_published'])
        
        if os.path.exists(os.path.join(DATA_PATH, 'skills.csv')):
            data = pl.read_csv(os.path.join(DATA_PATH, 'skills.csv'))
            data = pl.concat([data, new_data])
        else:
            data = new_data
        data.write_csv(os.path.join(DATA_PATH, 'skills.csv'))


# Забираем данные по вакансиям с hh

In [7]:
vacancy = VacancyClass()
skills = VacancySkillsClass()

In [8]:
%%time
vacancy.reset()
skills.reset()
breaks_count = 0

# going through all vacancies name
for element in vacancies_list:
#for element in [vacancies_list[0]]:
    new_ones = 0
    
    #getting amount of all vacancies
    try:
        answ = requests.get(addr, params={'text':element}, headers = head)
        if answ.status_code != 200:
            print('Error get info with ' + element + ' tag')
            break
    except:
        print('exception try to get list of vacansies for profession')
        breaks_count += 1
        print('end')
        break
        
    print(answ.url)
    time.sleep(1)
    
    info_tag = answ.json()
    amnt_pages = info_tag['pages']
    amnt_found = info_tag['found']
    
    # going through all pages
    for page in tqdm(range(amnt_pages)):
    #for page in tqdm(range(4)):
        try:
            answ = requests.get(addr, params={'text':element, 'page':page}, headers = head)
            if answ.status_code != 200:
                print('Error get info with ' + element + ' tag on page ' + str(page))
                break
        except:
            print(f'exception try to get next list of vacansies for {element}')
            breaks_count += 1
            if breaks_count > BREAK_STOP_LEVEL:
                break
            continue
            
        info_tag_page = answ.json()
        info_tag_page = info_tag_page['items']
        if len(info_tag_page) == 0:
            break

        #going through all vacancies on page
        for vac_idx in range( len(info_tag_page) ):
            if vacancy.check_id(info_tag_page[vac_idx]['id']):
                break
            
            try:
                #print(info_tag_page[vac]['id'])
                answ = requests.get(addr + '/' + info_tag_page[vac_idx]['id'], headers = head)
                if answ.status_code != 200:
                    print('Error get info about vacancia ' + info_tag_page[vac_idx]['id'] +\
                          'code ' + answ.status_code)
                    break  
            except:
                print('exception try to get vacancy description')
                breaks_count += 1
                if breaks_count > BREAK_STOP_LEVEL:
                    break
                continue
            vac = answ.json()
            
            vacancy.collect_data(vac)
            skills.collect_skills(vac)
            vacancy.add_id(info_tag_page[vac_idx]['id'])
            new_ones += 1

            time.sleep(0.37)
    
    print('Found ' + str(amnt_found) + ' vacancies with key words "' + element + '" with ' + str(new_ones) + ' not in list')

skills.saveskills()
vacancy.savevacancies()
print('\nDone')

https://api.hh.ru/vacancies?text=Data+scientist


  0%|          | 0/20 [00:00<?, ?it/s]

exception try to get vacancy description
exception try to get vacancy description
exception try to get vacancy description
exception try to get vacancy description
exception try to get vacancy description
exception try to get vacancy description
exception try to get vacancy description
exception try to get vacancy description
exception try to get vacancy description
exception try to get vacancy description
exception try to get vacancy description
exception try to get vacancy description
exception try to get vacancy description
exception try to get vacancy description
exception try to get vacancy description
exception try to get vacancy description
exception try to get vacancy description
Found 386 vacancies with key words "Data scientist" with 119 not in list
https://api.hh.ru/vacancies?text=Data+science


  0%|          | 0/37 [00:00<?, ?it/s]

exception try to get vacancy description
exception try to get vacancy description
exception try to get vacancy description
exception try to get vacancy description
exception try to get vacancy description
exception try to get vacancy description
exception try to get vacancy description
exception try to get vacancy description
exception try to get vacancy description
exception try to get vacancy description
exception try to get vacancy description
exception try to get vacancy description
exception try to get vacancy description
exception try to get vacancy description
exception try to get vacancy description
exception try to get vacancy description
exception try to get vacancy description
exception try to get vacancy description
exception try to get vacancy description
exception try to get vacancy description
exception try to get vacancy description
exception try to get vacancy description
exception try to get vacancy description
exception try to get vacancy description
exception try to

...

exception try to get vacancy description
exception try to get vacancy description
exception try to get vacancy description
exception try to get vacancy description
exception try to get vacancy description
exception try to get vacancy description
exception try to get vacancy description
exception try to get vacancy description
exception try to get vacancy description
exception try to get vacancy description
exception try to get vacancy description
Found 246 vacancies with key words "Ml-engeneer" with 18 not in list
https://api.hh.ru/vacancies?text=CV-engeneer


  0%|          | 0/1 [00:00<?, ?it/s]

exception try to get vacancy description
Found 10 vacancies with key words "CV-engeneer" with 3 not in list
https://api.hh.ru/vacancies?text=NLP


  0%|          | 0/23 [00:00<?, ?it/s]

exception try to get vacancy description
exception try to get vacancy description
exception try to get vacancy description
exception try to get vacancy description
exception try to get vacancy description
exception try to get vacancy description
exception try to get vacancy description
exception try to get vacancy description
exception try to get vacancy description
exception try to get vacancy description
exception try to get vacancy description
exception try to get vacancy description
exception try to get vacancy description
exception try to get vacancy description
exception try to get vacancy description
exception try to get vacancy description
exception try to get vacancy description
Found 443 vacancies with key words "NLP" with 13 not in list

Done
CPU times: total: 5.14 s
Wall time: 3min 30s
